# 15 — Convolutional Neural Network Foundations

In the previous notebook, we built a complete Multi-Layer Perceptron for classification.

Now we will move to one of the most important architectures in computer vision:

> **Convolutional Neural Networks — CNNs**

CNNs are designed to work naturally with image tensors. Instead of flattening an image immediately, they learn small spatial filters that move across the image and detect useful local patterns.

## In this notebook, we will study:

1. Why MLPs are limited for images
2. Image tensor structure
3. What is convolution?
4. Filters / kernels
5. Sliding-window intuition
6. Channels
7. Feature maps
8. `nn.Conv2d`
9. Kernel size
10. Stride
11. Padding
12. Output-size formula
13. Pooling
14. `nn.MaxPool2d`
15. Flattening convolution features
16. Building a small CNN
17. Shape reasoning through a CNN
18. Parameter counting
19. Ultrasound-image examples
20. Common CNN mistakes
21. Practice exercises

## Main Goal

By the end of this notebook, you should be able to reason through the shape:

$$
\boxed{(N,\ C,\ H,\ W)}
$$

at every stage of a CNN.

> **Never guess a CNN shape. Compute it carefully.**


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)


# 1. Why MLPs Are Limited for Images

Suppose a grayscale image has size:

$$
256\times256
$$

It contains:

$$
256\times256=65,536
$$

pixels.

If we flatten it and connect it to 512 hidden neurons, the first weight matrix contains:

$$
65,536\times512=\boxed{33,554,432}
$$

weights before counting biases.

A CNN is much more parameter-efficient because it reuses small filters across the image.


In [ ]:
input_features = 256 * 256
hidden_features = 512

print("Flattened features:", input_features)
print("Weights in first dense layer:", input_features * hidden_features)


# 2. Images Have Spatial Structure

An image is not just a long list of unrelated values.

Nearby pixels are spatially related. A local pattern such as an edge or texture is meaningful because of the arrangement of neighboring pixels.

CNNs preserve this structure and process local regions directly.


# 3. Image Tensor Structure in PyTorch

PyTorch usually stores a batch of images as:

$$
\boxed{(N,\ C,\ H,\ W)}
$$

$$
\begin{array}{|c|c|}
\hline
N & \text{Batch size} \\
\hline
C & \text{Channels} \\
\hline
H & \text{Height} \\
\hline
W & \text{Width} \\
\hline
\end{array}
$$

Example:

$$
(16,\ 1,\ 256,\ 256)
$$

means 16 grayscale images of size 256×256.


In [ ]:
images = torch.randn(16, 1, 256, 256)

print("Batch shape:", images.shape)
print("Batch size:", images.shape[0])
print("Channels:", images.shape[1])
print("Height:", images.shape[2])
print("Width:", images.shape[3])


# 4. Grayscale vs RGB

A grayscale image usually has:

$$
C=1
$$

An RGB image usually has:

$$
C=3
$$

So a batch might be:

$$
\text{Grayscale: }(N,1,H,W)
$$

$$
\text{RGB: }(N,3,H,W)
$$


In [ ]:
grayscale = torch.randn(8, 1, 64, 64)
rgb = torch.randn(8, 3, 64, 64)

print("Grayscale:", grayscale.shape)
print("RGB:", rgb.shape)


# 5. Ultrasound Images

Many ultrasound pipelines use one channel, so a batch may look like:

$$
\boxed{(16,\ 1,\ 256,\ 256)}
$$

However, always inspect the tensor that actually enters the model. Some image-loading pipelines may produce three channels even when the visible image looks grayscale.

The first convolution's `in_channels` must match the tensor's channel dimension.


# 6. What Is Convolution?

A convolution applies a small filter to local image regions.

At each position:

1. Take a local image patch
2. Multiply it element-by-element with the kernel
3. Sum the products
4. Produce one output value
5. Slide the kernel and repeat

The resulting spatial output is called a **feature map**.


# 7. A Small Sliding-Window Example

Consider this image:

$$
\begin{array}{|c|c|c|c|}
\hline
1 & 2 & 0 & 1 \\
\hline
3 & 1 & 2 & 2 \\
\hline
0 & 1 & 3 & 1 \\
\hline
2 & 2 & 1 & 0 \\
\hline
\end{array}
$$

and kernel:

$$
\begin{array}{|c|c|}
\hline
1 & 0 \\
\hline
0 & -1 \\
\hline
\end{array}
$$

The first patch is:

$$
\begin{array}{|c|c|}
\hline
1 & 2 \\
\hline
3 & 1 \\
\hline
\end{array}
$$

The first output value is:

$$
(1)(1)+(2)(0)+(3)(0)+(1)(-1)=\boxed{0}
$$


In [ ]:
image = torch.tensor([
    [1.0, 2.0, 0.0, 1.0],
    [3.0, 1.0, 2.0, 2.0],
    [0.0, 1.0, 3.0, 1.0],
    [2.0, 2.0, 1.0, 0.0]
])

kernel = torch.tensor([
    [1.0, 0.0],
    [0.0, -1.0]
])

output = torch.empty(3, 3)

for row in range(3):
    for col in range(3):
        patch = image[row:row+2, col:col+2]
        output[row, col] = (patch * kernel).sum()

print("Feature map:")
print(output)


# 8. A Small Technical Detail — PyTorch Uses Cross-Correlation

Strict mathematical convolution flips the kernel before sliding it.

`nn.Conv2d` performs the closely related operation commonly called **cross-correlation**, where the kernel is used without that explicit flip.

In deep learning, people still call this a convolution because the filter values are learned.

For shape reasoning, this distinction does not change anything.


# 9. What Is a Kernel / Filter?

A kernel is a small set of learnable weights.

Common spatial sizes include:

$$
1\times1,\qquad3\times3,\qquad5\times5
$$

Early CNN layers often learn local patterns resembling edges or textures. Deeper layers combine simpler features into more task-specific representations.


# 10. Weight Sharing

The same kernel is reused at many spatial positions.

This is called:

> **Weight sharing**

A pattern detector does not need separate weights for the top-left, center, and bottom-right of the image.

This is one reason CNNs use far fewer parameters than fully connected image models.


# 11. What Is a Feature Map?

One learned output filter produces one output channel, also called a feature map.

If a convolution has:

$$
out\_channels=16
$$

then it learns 16 output filters and produces 16 output feature maps.

So:

> **`out_channels` = number of learned output filters = number of output feature maps.**


# 12. Channels in Convolution

Suppose the input is grayscale:

$$
C_{in}=1
$$

and the layer produces:

$$
C_{out}=8
$$

feature maps.

Then:

$$
(N,1,H,W)
\rightarrow
(N,8,H_{out},W_{out})
$$


# 13. RGB Filters Span All Input Channels

For RGB input:

$$
C_{in}=3
$$

A 3×3 filter spans all three channels.

One output filter therefore has shape:

$$
\boxed{(3,3,3)}
$$

where the first 3 represents input channels and the other two are spatial dimensions.


# 14. `nn.Conv2d`

PyTorch provides:

```python
nn.Conv2d(
    in_channels,
    out_channels,
    kernel_size,
    stride=1,
    padding=0
)
```

Example:

```python
nn.Conv2d(1, 8, kernel_size=3)
```

means:

- 1 input channel
- 8 output channels
- 3×3 kernels


In [ ]:
conv = nn.Conv2d(
    in_channels=1,
    out_channels=8,
    kernel_size=3
)

print(conv)
print("Weight shape:", conv.weight.shape)
print("Bias shape:", conv.bias.shape)


# 15. Conv2d Weight Shape

For:

```python
nn.Conv2d(1, 8, kernel_size=3)
```

PyTorch stores weights as:

$$
\boxed{(8,\ 1,\ 3,\ 3)}
$$

$$
\begin{array}{|c|c|}
\hline
8 & \text{Output filters} \\
\hline
1 & \text{Input channels per filter} \\
\hline
3 & \text{Kernel height} \\
\hline
3 & \text{Kernel width} \\
\hline
\end{array}
$$


# 16. Applying Conv2d

Input:

$$
(4,\ 1,\ 32,\ 32)
$$

Using:

```python
Conv2d(1, 8, kernel_size=3)
```

with default stride 1 and padding 0 produces:

$$
(4,\ 8,\ 30,\ 30)
$$


In [ ]:
x = torch.randn(4, 1, 32, 32)
conv = nn.Conv2d(1, 8, kernel_size=3)
y = conv(x)

print("Input:", x.shape)
print("Output:", y.shape)


# 17. Convolution Output-Size Formula

For one spatial dimension:

$$
\boxed{
O=\left\lfloor
\frac{I+2P-D(K-1)-1}{S}+1
\right\rfloor
}
$$

where:

$$
\begin{array}{|c|c|}
\hline
I & \text{Input size} \\
\hline
K & \text{Kernel size} \\
\hline
S & \text{Stride} \\
\hline
P & \text{Padding} \\
\hline
D & \text{Dilation} \\
\hline
O & \text{Output size} \\
\hline
\end{array}
$$

For the common case $D=1$:

$$
\boxed{
O=\left\lfloor
\frac{I+2P-K}{S}
\right\rfloor+1
}
$$


# 18. Example — No Padding

For:

$$
I=32,\quad K=3,\quad S=1,\quad P=0
$$

$$
O=
\frac{32-3}{1}+1
=\boxed{30}
$$

So:

$$
32\times32
\rightarrow
30\times30
$$


# 19. Kernel Size

`kernel_size=3` means a 3×3 spatial kernel.

`kernel_size=5` means a 5×5 spatial kernel.

Larger kernels examine larger local regions but use more parameters and computation.


In [ ]:
x = torch.randn(1, 1, 32, 32)

conv3 = nn.Conv2d(1, 4, kernel_size=3)
conv5 = nn.Conv2d(1, 4, kernel_size=5)

print("3x3 output:", conv3(x).shape)
print("5x5 output:", conv5(x).shape)


# 20. Stride

Stride controls how far the kernel moves each step.

$$
stride=1
$$

means move one pixel at a time.

$$
stride=2
$$

means move two pixels at a time.

Larger stride reduces spatial resolution.


In [ ]:
x = torch.randn(1, 1, 32, 32)

conv_s1 = nn.Conv2d(1, 4, 3, stride=1)
conv_s2 = nn.Conv2d(1, 4, 3, stride=2)

print("Stride 1:", conv_s1(x).shape)
print("Stride 2:", conv_s2(x).shape)


# 21. Stride-2 Calculation

For:

$$
I=32,\quad K=3,\quad S=2,\quad P=0
$$

$$
O=
\left\lfloor
\frac{32-3}{2}
\right\rfloor+1
$$

$$
=14+1
$$

$$
=\boxed{15}
$$


# 22. Padding

Padding adds values around the image boundary before convolution.

Zero padding is common.

Padding helps control output size and allows border pixels to participate in more kernel positions.


# 23. Padding Example — Preserve Spatial Size

For:

$$
I=32,\quad K=3,\quad S=1,\quad P=1
$$

$$
O=
\frac{32+2-3}{1}+1
=\boxed{32}
$$

So a 3×3 convolution with padding 1 and stride 1 preserves height and width.


In [ ]:
x = torch.randn(4, 1, 32, 32)
conv = nn.Conv2d(1, 8, 3, stride=1, padding=1)

y = conv(x)

print("Input:", x.shape)
print("Output:", y.shape)


# 24. A Very Common 3×3 Pattern

A common convolution is:

```python
nn.Conv2d(
    in_channels,
    out_channels,
    kernel_size=3,
    stride=1,
    padding=1
)
```

It preserves spatial dimensions while changing the channel count.

Example:

$$
(32,3,64,64)
\rightarrow
(32,16,64,64)
$$


In [ ]:
conv = nn.Conv2d(3, 16, kernel_size=3, padding=1)
x = torch.randn(32, 3, 64, 64)

print(conv(x).shape)


# 25. Conv2d Parameter Count

For a standard convolution with bias:

$$
\boxed{
C_{out}\times C_{in}\times K_H\times K_W+C_{out}
}
$$

For:

$$
C_{in}=1,\ C_{out}=8,\ K=3
$$

$$
8\times1\times3\times3+8
=72+8
=\boxed{80}
$$


In [ ]:
conv = nn.Conv2d(1, 8, kernel_size=3)

print(
    "Parameters:",
    sum(p.numel() for p in conv.parameters())
)


# 26. Compare Convolution With a Dense Layer

For a 64×64 grayscale image:

$$
64\times64=4096
$$

A dense layer from 4096 inputs to 8 outputs has:

$$
4096\times8+8
$$

parameters.

A 3×3 convolution from 1 channel to 8 channels has only:

$$
1\times8\times3\times3+8
$$

parameters.


In [ ]:
linear = nn.Linear(64 * 64, 8)
conv = nn.Conv2d(1, 8, kernel_size=3)

print("Linear parameters:", sum(p.numel() for p in linear.parameters()))
print("Conv parameters:", sum(p.numel() for p in conv.parameters()))


# 27. What Is Pooling?

Pooling reduces spatial resolution.

A common choice is **max pooling**.

For a 2×2 patch:

$$
\begin{array}{|c|c|}
\hline
1 & 7 \\
\hline
3 & 2 \\
\hline
\end{array}
$$

max pooling returns:

$$
\boxed{7}
$$


# 28. Why Pooling?

Pooling can:

- Reduce height and width
- Reduce memory usage
- Reduce computation
- Increase the effective receptive field
- Keep strong local activations

A common choice is 2×2 max pooling with stride 2.


# 29. `nn.MaxPool2d`

Example:

```python
nn.MaxPool2d(
    kernel_size=2,
    stride=2
)
```

A 32×32 feature map becomes approximately 16×16.


In [ ]:
pool = nn.MaxPool2d(2, stride=2)

x = torch.randn(4, 8, 32, 32)
y = pool(x)

print("Input:", x.shape)
print("Output:", y.shape)


# 30. Pooling Preserves Channel Count

Max pooling changes spatial dimensions but normally keeps the number of channels unchanged.

$$
(4,8,32,32)
\rightarrow
\boxed{(4,8,16,16)}
$$

The 8 channels remain 8 channels.


# 31. Manual Max-Pooling Example

Consider:

$$
\begin{array}{|c|c|c|c|}
\hline
1 & 5 & 2 & 4 \\
\hline
3 & 2 & 8 & 1 \\
\hline
0 & 6 & 3 & 7 \\
\hline
4 & 1 & 2 & 9 \\
\hline
\end{array}
$$

Using 2×2 max pooling with stride 2 produces:

$$
\begin{array}{|c|c|}
\hline
5 & 8 \\
\hline
6 & 9 \\
\hline
\end{array}
$$


In [ ]:
x = torch.tensor([
    [[
        [1.0, 5.0, 2.0, 4.0],
        [3.0, 2.0, 8.0, 1.0],
        [0.0, 6.0, 3.0, 7.0],
        [4.0, 1.0, 2.0, 9.0]
    ]]
])

pool = nn.MaxPool2d(2)
print(pool(x))


# 32. A Classic CNN Block

A very common pattern is:

$$
\boxed{
Conv
\rightarrow
ReLU
\rightarrow
Pool
}
$$

Example:

```python
Conv2d(1, 8, 3, padding=1)
ReLU()
MaxPool2d(2)
```


In [ ]:
block = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2)
)

x = torch.randn(16, 1, 64, 64)
print(block(x).shape)


# 33. Shape Through One CNN Block

Input:

$$
(16,1,64,64)
$$

After convolution:

$$
(16,8,64,64)
$$

After ReLU:

$$
(16,8,64,64)
$$

After pooling:

$$
\boxed{(16,8,32,32)}
$$


# 34. A Second CNN Block

Suppose the second block is:

```python
Conv2d(8, 16, 3, padding=1)
ReLU()
MaxPool2d(2)
```

Then:

$$
(16,8,32,32)
\rightarrow
(16,16,32,32)
\rightarrow
\boxed{(16,16,16,16)}
$$


In [ ]:
block2 = nn.Sequential(
    nn.Conv2d(8, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2)
)

x = torch.randn(16, 8, 32, 32)
print(block2(x).shape)


# 35. A Common CNN Pattern

CNNs often reduce spatial size while increasing channel depth.

For example:

$$
(1,64,64)
$$

$$
\downarrow
$$

$$
(8,32,32)
$$

$$
\downarrow
$$

$$
(16,16,16)
$$

The network gradually converts raw pixels into richer learned features.


# 36. Flattening Convolution Features

Suppose the final feature tensor is:

$$
(N,16,16,16)
$$

Each sample has:

$$
16\times16\times16=\boxed{4096}
$$

features.

Flatten while preserving the batch dimension:

$$
(N,16,16,16)
\rightarrow
(N,4096)
$$


In [ ]:
x = torch.randn(32, 16, 16, 16)
flat = torch.flatten(x, start_dim=1)

print("Before:", x.shape)
print("After:", flat.shape)


# 37. Why `start_dim=1`?

Dimension 0 is the batch dimension.

Using:

```python
torch.flatten(x, start_dim=1)
```

preserves the batch size and combines:

$$
C,\ H,\ W
$$

into one feature dimension.


# 38. `nn.Flatten`

PyTorch also provides:

```python
nn.Flatten()
```

For a standard image batch, it preserves the batch dimension by default.


In [ ]:
flatten = nn.Flatten()
x = torch.randn(32, 16, 16, 16)

print(flatten(x).shape)


# 39. Building a Small CNN

We will build a CNN for grayscale 64×64 images and 3 output classes.

Architecture:

$$
Input
\rightarrow
Conv
\rightarrow
ReLU
\rightarrow
Pool
\rightarrow
Conv
\rightarrow
ReLU
\rightarrow
Pool
\rightarrow
Flatten
\rightarrow
Linear
$$


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.flatten = nn.Flatten()

        self.classifier = nn.Linear(
            16 * 16 * 16,
            num_classes
        )

    def forward(self, x):
        x = self.features(x)
        x = self.flatten(x)
        x = self.classifier(x)
        return x

model = SmallCNN(num_classes=3)
print(model)


# 40. Shape Reasoning Through the Small CNN

Input:

$$
(N,1,64,64)
$$

First convolution:

$$
(N,8,64,64)
$$

First pool:

$$
(N,8,32,32)
$$

Second convolution:

$$
(N,16,32,32)
$$

Second pool:

$$
(N,16,16,16)
$$

Flatten:

$$
(N,4096)
$$

Classifier:

$$
\boxed{(N,3)}
$$


# 41. Verify Every Shape With Code


In [ ]:
x = torch.randn(4, 1, 64, 64)

conv1 = model.features[0](x)
relu1 = model.features[1](conv1)
pool1 = model.features[2](relu1)
conv2 = model.features[3](pool1)
relu2 = model.features[4](conv2)
pool2 = model.features[5](relu2)
flat = model.flatten(pool2)
logits = model.classifier(flat)

print("Input:", x.shape)
print("Conv1:", conv1.shape)
print("ReLU1:", relu1.shape)
print("Pool1:", pool1.shape)
print("Conv2:", conv2.shape)
print("ReLU2:", relu2.shape)
print("Pool2:", pool2.shape)
print("Flatten:", flat.shape)
print("Logits:", logits.shape)


# 42. Final Output Is Logits

For 3-class classification, the final output is:

$$
(N,3)
$$

These are raw logits.

They can be passed directly to:

`nn.CrossEntropyLoss()`

Do not apply softmax before the loss.


In [ ]:
x = torch.randn(4, 1, 64, 64)
logits = model(x)

print("Logits shape:", logits.shape)


# 43. Parameter Counting in the CNN

Let's inspect every trainable parameter.


In [ ]:
for name, parameter in model.named_parameters():
    print(
        name,
        tuple(parameter.shape),
        "numel =",
        parameter.numel()
    )

print(
    "Total parameters:",
    sum(p.numel() for p in model.parameters())
)


# 44. The Dense Classifier Can Dominate Parameter Count

After the convolution blocks, each sample has:

$$
4096
$$

features.

The final classifier for 3 classes has:

$$
4096\times3+3
$$

parameters.

For larger images, flattening too early can create a very large dense classifier.


# 45. Input Resolution Affects Flattened Size

Our `SmallCNN` assumes 64×64 input.

If the input becomes 80×80, the convolutional output has a different spatial size, so the classifier's expected `in_features` no longer matches.


In [ ]:
wrong_size = torch.randn(4, 1, 80, 80)
features_only = model.features(wrong_size)
flat = torch.flatten(features_only, start_dim=1)

print("Feature shape:", features_only.shape)
print("Flattened shape:", flat.shape)
print("Classifier expects:", model.classifier.in_features)


# 46. Inspect Flatten Size With a Dummy Tensor

A useful debugging technique is to pass a dummy tensor through the feature extractor before defining or checking the classifier.


In [ ]:
dummy = torch.zeros(1, 1, 64, 64)

with torch.no_grad():
    feature_output = model.features(dummy)

flattened_size = feature_output[0].numel()

print("Feature output:", feature_output.shape)
print("Flattened size:", flattened_size)


# 47. A CNN That Computes Its Flatten Size

For educational models, we can compute the flattened size automatically using a dummy input.


In [ ]:
class FlexibleSmallCNN(nn.Module):
    def __init__(self, image_size=64, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, image_size, image_size)
            feature_output = self.features(dummy)
            flattened_size = feature_output[0].numel()

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_size, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

flex_model = FlexibleSmallCNN(image_size=64, num_classes=3)
print(flex_model)


# 48. RGB Version

For RGB input, the first convolution must usually use:

$$
in\_channels=3
$$


In [ ]:
rgb_conv = nn.Conv2d(3, 8, 3, padding=1)
rgb_batch = torch.randn(4, 3, 64, 64)

print(rgb_conv(rgb_batch).shape)


# 49. Channel-Mismatch Error

If the input is:

$$
(N,1,H,W)
$$

but the first convolution expects:

```python
nn.Conv2d(3, 16, 3)
```

then the layer expects three channels but receives one.

Always verify:

$$
\boxed{input.shape[1]=first\_conv.in\_channels}
$$


# 50. Ultrasound CNN Example

Suppose an ultrasound batch has:

$$
\boxed{(8,1,256,256)}
$$

Apply:

```python
Conv2d(1, 16, 3, padding=1)
ReLU()
MaxPool2d(2)
```

Shape flow:

$$
(8,1,256,256)
\rightarrow
(8,16,256,256)
\rightarrow
\boxed{(8,16,128,128)}
$$


In [ ]:
ultrasound = torch.randn(8, 1, 256, 256)

ultrasound_block = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2)
)

print(ultrasound_block(ultrasound).shape)


# 51. Deeper Ultrasound Shape Example

Starting from:

$$
(8,1,256,256)
$$

three blocks can give:

$$
(8,16,128,128)
$$

$$
\downarrow
$$

$$
(8,32,64,64)
$$

$$
\downarrow
$$

$$
\boxed{(8,64,32,32)}
$$


In [ ]:
ultrasound_features = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(16, 32, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(32, 64, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2)
)

x = torch.randn(8, 1, 256, 256)
print(ultrasound_features(x).shape)


# 52. Flattening the Ultrasound Features

After three pooling stages:

$$
(8,64,32,32)
$$

The flattened feature size is:

$$
64\times32\times32=\boxed{65,536}
$$

That is still large.

This is why practical CNNs often keep reducing spatial dimensions before the final classifier.


In [ ]:
features = ultrasound_features(x)
flat = torch.flatten(features, start_dim=1)

print("Feature shape:", features.shape)
print("Flattened shape:", flat.shape)


# 53. Receptive Field Intuition

An early feature responds to a small local region.

As we stack convolutions and pooling operations, deeper features indirectly depend on larger parts of the original image.

The region of the input that can influence a feature is called its:

> **Receptive field**

We will study this more deeply later.


# 54. Learnable vs Non-Learnable Operations

$$
\begin{array}{|c|c|}
\hline
\textbf{Layer} & \textbf{Learnable Parameters?} \\
\hline
Conv2d & Yes \\
\hline
ReLU & No \\
\hline
MaxPool2d & No \\
\hline
Linear & Yes \\
\hline
\end{array}
$$


# 55. Shape Rules to Memorize

## Conv2d

$$
(N,C_{in},H,W)
\rightarrow
(N,C_{out},H_{out},W_{out})
$$

## ReLU

Shape unchanged.

## MaxPool2d

Channels usually unchanged; spatial dimensions shrink.

## Flatten

$$
(N,C,H,W)
\rightarrow
(N,C\times H\times W)
$$

## Linear

Changes the final feature dimension to `out_features`.


# 56. Shape Example 1

Input:

$$
(32,1,28,28)
$$

Layer:

```python
Conv2d(1, 8, 3, padding=1)
```

Output:

$$
\boxed{(32,8,28,28)}
$$

Then:

```python
MaxPool2d(2)
```

Output:

$$
\boxed{(32,8,14,14)}
$$


# 57. Shape Example 2

Input:

$$
(16,3,64,64)
$$

Layer:

```python
Conv2d(3, 32, kernel_size=5)
```

With stride 1 and padding 0:

$$
64-5+1=60
$$

Output:

$$
\boxed{(16,32,60,60)}
$$


# 58. Shape Example 3 — Stride 2

Input:

$$
(8,16,32,32)
$$

Layer:

```python
Conv2d(16, 32, 3, stride=2, padding=1)
```

Spatial output:

$$
\left\lfloor
\frac{32+2-3}{2}
\right\rfloor+1
=16
$$

Output:

$$
\boxed{(8,32,16,16)}
$$


In [ ]:
x = torch.randn(8, 16, 32, 32)
conv = nn.Conv2d(16, 32, 3, stride=2, padding=1)

print(conv(x).shape)


# 59. Common Mistake — Wrong Channel Order

PyTorch expects:

$$
N,C,H,W
$$

Some libraries use:

$$
N,H,W,C
$$

If necessary, a tensor can be rearranged with:

```python
x = x.permute(0, 3, 1, 2)
```

But never permute blindly. First identify what every dimension currently represents.


# 60. Common Mistake — Wrong `in_channels`

If your tensor is:

$$
(N,1,H,W)
$$

then the first convolution normally needs `in_channels=1`.

If your tensor is:

$$
(N,3,H,W)
$$

then it normally needs `in_channels=3`.


# 61. Common Mistake — Wrong Flatten Size

A frequent error is defining:

```python
nn.Linear(wrong_number, num_classes)
```

If the feature-map shape before flattening is not what you expected, the linear layer fails.

Always inspect the tensor immediately before flattening.


# 62. Common Mistake — Flattening the Batch Dimension

Wrong for a batch:

```python
x = torch.flatten(x)
```

This collapses every dimension.

Usually use:

```python
x = torch.flatten(x, start_dim=1)
```

or:

```python
nn.Flatten()
```


# 63. Common Mistake — Forgetting Padding Effects

A 3×3 convolution with padding 0 shrinks spatial dimensions.

A 3×3 convolution with padding 1 and stride 1 preserves them.

Do not assume every convolution keeps height and width unchanged.


# 64. Common Mistake — Assuming Pooling Changes Channels

For standard max pooling:

$$
(16,32,64,64)
\rightarrow
\boxed{(16,32,32,32)}
$$

The channel count stays 32.


# 65. Common Mistake — Ignoring Image Resolution

A CNN whose dense classifier assumes 64×64 input may fail if preprocessing later produces 80×80 or 256×256 images.

Always reason from the actual tensor size produced by your preprocessing pipeline.


# 66. Common Mistake — Softmax Before `CrossEntropyLoss`

For multi-class classification:

```python
logits = model(images)
loss = criterion(logits, targets)
```

when:

```python
criterion = nn.CrossEntropyLoss()
```

Use raw logits.


# 67. CNN Debugging Checklist

If a CNN throws a shape error, inspect:

1. Input shape
2. Dimension order
3. `in_channels`
4. `out_channels`
5. Kernel size
6. Stride
7. Padding
8. Pooling settings
9. Feature shape before flattening
10. Flattened size
11. Linear `in_features`
12. Number of output logits

> Print the shape after every major layer.


In [ ]:
debug_model = SmallCNN(num_classes=3)
x = torch.randn(2, 1, 64, 64)

print("Input:", x.shape)

for index, layer in enumerate(debug_model.features):
    x = layer(x)
    print(f"After feature layer {index}:", x.shape)

x = debug_model.flatten(x)
print("After flatten:", x.shape)

x = debug_model.classifier(x)
print("After classifier:", x.shape)


# 68. Practice Exercises

Try solving these before looking at the solutions.

## Exercise 1

Input:

$$
(16,1,28,28)
$$

Apply:

```python
Conv2d(1, 8, kernel_size=3, padding=1)
```

What is the output shape?

## Exercise 2

Take Exercise 1 output and apply:

```python
MaxPool2d(2)
```

What is the output shape?

## Exercise 3

Input:

$$
(8,3,64,64)
$$

Apply:

```python
Conv2d(3, 16, kernel_size=5)
```

What is the output shape?

## Exercise 4

How many parameters are in:

```python
Conv2d(3, 16, kernel_size=3)
```

including bias?

## Exercise 5

Build two convolution blocks:

$$
1\rightarrow8\rightarrow16
$$

Each block should use:

- 3×3 convolution
- Padding 1
- ReLU
- 2×2 max pooling

## Exercise 6

For input:

$$
(32,1,64,64)
$$

find the output after both blocks.

## Exercise 7

Flatten the Exercise 6 result. What is the feature size per sample?

## Exercise 8

Add a linear classifier for 4 classes.

## Exercise 9

Modify the first convolution to accept RGB images.

## Exercise 10

For ultrasound input:

$$
(8,1,256,256)
$$

apply three 2×2 pooling operations. What is the final spatial size?


# 69. Shape Reasoning Challenges

Answer before running code.

## Challenge 1

Input:

$$
(64,3,224,224)
$$

Apply:

```python
Conv2d(3, 32, 3, stride=1, padding=1)
```

Output shape?

## Challenge 2

Then apply:

```python
MaxPool2d(2)
```

Output shape?

## Challenge 3

Then apply:

```python
Conv2d(32, 64, 3, padding=1)
```

Output shape?

## Challenge 4

Then apply another:

```python
MaxPool2d(2)
```

Output shape?

## Challenge 5

How many flattened features per sample remain after Challenge 4?

## Challenge 6

Why can convolution parameter count stay relatively small even for large images?

## Challenge 7

Why does a 3×3 RGB convolutional filter span all three input channels?

## Challenge 8

Why can a CNN preserve spatial structure better than immediately flattening an image into an MLP?


# 70. Exercise Solutions


In [ ]:
# Exercise 1
x1 = torch.randn(16, 1, 28, 28)
conv1 = nn.Conv2d(1, 8, 3, padding=1)
out1 = conv1(x1)
print("Exercise 1:", out1.shape)

# Exercise 2
pool = nn.MaxPool2d(2)
out2 = pool(out1)
print("Exercise 2:", out2.shape)

# Exercise 3
x3 = torch.randn(8, 3, 64, 64)
conv3 = nn.Conv2d(3, 16, 5)
out3 = conv3(x3)
print("Exercise 3:", out3.shape)

# Exercise 4
conv4 = nn.Conv2d(3, 16, 3)
print("Exercise 4:", sum(p.numel() for p in conv4.parameters()))

# Exercise 5
exercise_cnn = nn.Sequential(
    nn.Conv2d(1, 8, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(8, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2)
)
print("Exercise 5:")
print(exercise_cnn)

# Exercise 6
x6 = torch.randn(32, 1, 64, 64)
out6 = exercise_cnn(x6)
print("Exercise 6:", out6.shape)

# Exercise 7
flat7 = torch.flatten(out6, start_dim=1)
print("Exercise 7:", flat7.shape)

# Exercise 8
classifier8 = nn.Linear(flat7.shape[1], 4)
logits8 = classifier8(flat7)
print("Exercise 8:", logits8.shape)

# Exercise 9
rgb_first_conv = nn.Conv2d(3, 8, 3, padding=1)
print("Exercise 9:", rgb_first_conv)

# Exercise 10
spatial = 256
for _ in range(3):
    spatial //= 2
print("Exercise 10 final spatial size:", spatial)


# 71. Key Takeaways

In this notebook, we learned:

- Why MLPs are inefficient for raw images
- Image tensor structure
- `N, C, H, W`
- Grayscale and RGB channels
- Convolution intuition
- Cross-correlation detail
- Kernels / filters
- Sliding windows
- Weight sharing
- Feature maps
- Input and output channels
- `nn.Conv2d`
- Conv2d weight shapes
- Kernel size
- Stride
- Padding
- Convolution output-size formula
- Max pooling
- `nn.MaxPool2d`
- Flattening convolution features
- `nn.Flatten`
- Building a small CNN
- Parameter counting
- Ultrasound CNN shapes
- Common CNN mistakes
- CNN debugging

The central pattern is:

$$
\boxed{
Conv
\rightarrow
ReLU
\rightarrow
Pool
}
$$

Repeated CNN blocks transform:

$$
\boxed{
Image
\rightarrow
Feature\ Maps
\rightarrow
Higher\text{-}Level\ Features
\rightarrow
Classifier
}
$$


# 72. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. Why can an MLP be inefficient for images?
2. What does `NCHW` mean?
3. What is a convolutional kernel?
4. What does sliding-window computation mean?
5. What is weight sharing?
6. What is a feature map?
7. What does `in_channels` mean?
8. What does `out_channels` mean?
9. What shape does a Conv2d weight tensor have?
10. What does kernel size control?
11. What does stride control?
12. What does padding control?
13. What is the convolution output-size formula?
14. What does MaxPool2d do?
15. Does max pooling change channel count?
16. Does ReLU change shape?
17. Why do we flatten before a linear classifier?
18. Why must flattening preserve the batch dimension?
19. Why can changing image resolution break a dense classifier?
20. How do you modify the first convolution for RGB input?
21. How would a grayscale ultrasound batch usually be arranged?
22. Why is printing shapes after major CNN layers such a useful debugging habit?


# Next Notebook

# 16 — Training a CNN for Image Classification

In the next notebook, we will study:

- Creating an image-classification dataset
- CNN input pipelines
- Image normalization
- Data augmentation
- Train / validation / test loaders
- Building a practical CNN
- `CrossEntropyLoss`
- CNN training loop
- Validation loop
- Accuracy
- Confusion matrix
- Saving the best CNN
- Overfitting in CNNs
- Improving CNN performance
- Preparing the pipeline for real ultrasound data
